In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

from src import config as cfg, data, adaboost, tuning

splits = {name: data.get_split(name) for name in cfg.DATASETS}


Spambase: 4601 rows, 391 duplicates removed


In [4]:
def brute_force(X,y,p):
    return min(p@(adaboost.stump_predict(X,j,tau,s) != y) for j in range (X.shape[1]) for tau in np.r_[-np.inf, X[:, j]] for s in (1,-1))

rng = np.random.default_rng(0)
X, y = splits["spambase"]["X_train"][:60, :8], splits["spambase"]["y_train"][:60]
p = rng.random(60)
p /= p.sum()
j,tau, s, eps = adaboost.best_stump(*adaboost.presort(X), p,y)

print(f"find eps = {eps:.6f}, recalculated = {p @ (adaboost.stump_predict(X, j, tau, s) != y):.6f}, "
      f"brute force = {brute_force(X, y, p):.6f}")

find eps = -0.500000, recalculated = 0.750727, brute force = 0.159590


In [7]:
for name, s in splits.items():
    X, y = s["X_train"], s["y_train"]
    t0 = time.perf_counter()
    model = adaboost.fit(X,y, cfg.T_MAX, store_weights=True)
    elapsed = time.perf_counter() -t0
    W, eps = model["weights"], model["eps"]
    H = np.array([adaboost.stump_predict(X,j,tau,sg) for j, tau, sg in zip(model["feature"], model["threshold"], model["polarity"])])
    F = adaboost.staged_scores(model, X)
    bound = np.cumprod(2*np.sqrt(eps*(1-eps)))
    train_err = np.mean(adaboost.sgn(F) != y, axis=1)
    print(f"\n{name}: T = {len(eps)}, fit in {elapsed:.1f}s")
    print(f"  |eps recalculated - eps|          max = {np.abs((W[:-1] * (H != y)).sum(1) - eps).max():.1e}")
    print(f"  |error of h_t below p^(t+1) - 1/2| max = {np.abs((W[1:] * (H != y)).sum(1) - 0.5).max():.1e}")
    print(f"  |exp-loss of F/2 - bound|         max = {np.abs(np.exp(-0.5 * y * F).mean(1) - bound).max():.1e}")
    print(f"  training error <= bound for each t: {np.all(train_err <= bound)}, "
          f"final training error = {train_err[-1]:.3f}, eps max = {eps.max():.3f}")


spambase: T = 1, fit in 0.0s


C:\Users\cola0\AppData\Local\Temp\ipykernel_18120\673178101.py:9: RuntimeWarning: invalid value encountered in sqrt
  bound = np.cumprod(2*np.sqrt(eps*(1-eps)))


ValueError: zero-size array to reduction operation maximum which has no identity